# Raspador de Sites Gov.br Comprometidos

Este notebook foi desenvolvido para a investigacao jornalistica publicada na **Folha de S.Paulo** sobre hackers que invadiram sites governamentais brasileiros (.gov.br) inserindo links de apostas online, pornografia e outros conteudos ilicitos.

**Materia:** [Hackers poem Tigrinho, links suspeitos e termos sexuais em sites de governos](https://www1.folha.uol.com.br/mercado/2024/06/hackers-poem-tigrinho-links-suspeitos-e-termos-sexuais-em-sites-de-governos.shtml)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labintrieri/sites-apostas-govbr/blob/main/rob%C3%B4_raspador_final.ipynb)

---

## Metodologia

1. **Buscas no Google** com o operador `inurl:gov.br` combinado com termos suspeitos
2. **Raspagem dos resultados** usando BeautifulSoup
3. **Extracao de dados**: titulo, URL e orgao responsavel
4. **Exportacao** para CSV/Excel para analise

## 1. Importacao das bibliotecas

- **requests**: Para fazer requisicoes HTTP ao Google
- **BeautifulSoup**: Para parsear o HTML dos resultados
- **pandas**: Para organizar os dados em DataFrames
- **tqdm**: Para mostrar barra de progresso
- **time**: Para adicionar delays entre requisicoes (evitar bloqueio)

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import csv
from tqdm import tqdm
import pandas as pd

## 2. Lista de termos de busca

Cada termo e combinado com o operador `inurl:gov.br` para buscar apenas em dominios governamentais.

**Categorias de termos utilizados:**
- **Apostas**: bet, slots, fortune tiger, foguetinho, cassino online, jackpots
- **Conteudo adulto**: pornhub, xxxvideos, porno, nude
- **Exploracao**: termos relacionados a abuso e exploracao
- **Golpes**: lucre, ganhar dinheiro com prostituicao

In [ ]:
# Lista de URLs de busca no Google
# Cada URL contem um termo diferente combinado com inurl:gov.br
urls = [
    {"name": "novinha", "url": "https://www.google.com/search?q=%22novinha%22+inurl%3Agov.br"},
    {"name": "lucre", "url": "https://www.google.com/search?q=%22lucre%22+inurl%3Agov.br"},
    {"name": "ganhar dinheiro com prostituicao", "url": "https://www.google.com/search?q=%22ganhar+dinheiro+com+prostitui%C3%A7%C3%A3o%22+inurl%3Agov.br"},
    {"name": "putaria", "url": "https://www.google.com/search?q=%22putaria%22+inurl%3Agov.br"},
    {"name": "conteudo abuso menores", "url": "https://www.google.com/search?q=%22buceta+de+15+anos%22+inurl%3Agov.br"},
    {"name": "xxxvideos", "url": "https://www.google.com/search?q=%22xxxvideos%22+inurl%3Agov.br"},
    {"name": "porno", "url": "https://www.google.com/search?q=%22porno%22+inurl%3Agov.br"},
    {"name": "bet", "url": "https://www.google.com/search?q=%22bet%22+inurl%3Agov.br"},
    {"name": "nude", "url": "https://www.google.com/search?q=%22nude%22+inurl%3Agov.br"},
    {"name": "fortune tiger", "url": "https://www.google.com/search?q=%22fortune+tiger%22+inurl%3Agov.br"},
    {"name": "slots", "url": "https://www.google.com/search?q=%22slots%22+inurl%3Agov.br"},
    {"name": "foguetinho", "url": "https://www.google.com/search?q=%22foguetinho%22+inurl%3Agov.br"},
    {"name": "cassino online", "url": "https://www.google.com/search?q=%22cassino+online%22+inurl%3Agov.br"},
    {"name": "jackpots", "url": "https://www.google.com/search?q=%22jackpots%22+inurl%3Agov.br"},
    {"name": "pornhub", "url": "https://www.google.com/search?q=%22pornhub%22+inurl%3Agov.br"}
]

print(f"Total de termos para busca: {len(urls)}")

## 3. Funcao de raspagem

O codigo abaixo faz a raspagem dos resultados do Google:
1. Para cada URL de busca, faz uma requisicao HTTP
2. Parseia o HTML com BeautifulSoup
3. Extrai o titulo (link azul), URL e orgao governamental
4. Adiciona um delay de 1 segundo entre requisicoes para evitar bloqueio

In [ ]:
# Executa a raspagem e armazena resultados em uma lista
final = []

for item in tqdm(urls, desc="Raspando termos"):
    try:
        result = requests.get(item["url"], timeout=30)
        soup = BeautifulSoup(result.text, 'html.parser')
        
        # Encontra as divs com os resultados de busca
        divs = soup.find_all('div', class_='DnJfK')
        
        for div in divs:
            try:
                # Extrai o titulo (link azul)
                link_azul = div.find('div', class_='BNeawe vvjwJb AP7Wnd').text
                
                # Extrai a URL e identifica o orgao
                url = div.find('div', class_='BNeawe UPmit AP7Wnd lRVwie').text.replace(' › ', '/')
                orgao = url.split('/')[0].strip()
                
                final.append([item.get("name", "Sem nome"), link_azul, orgao, url])
            except AttributeError:
                continue  # Pula resultados mal formatados
                
        time.sleep(1)  # Delay para evitar bloqueio
        
    except requests.RequestException as e:
        print(f"Erro ao buscar {item['name']}: {e}")

print(f"\nTotal de resultados coletados: {len(final)}")

## 4. Criacao do DataFrame

Converte os resultados para um DataFrame do Pandas para facilitar a analise e exportacao.

In [ ]:
# Cria DataFrame com os resultados
df = pd.DataFrame(final, columns=['termo', 'titulo', 'orgao', 'url'])

# Exibe as primeiras linhas
print(f"Shape do DataFrame: {df.shape}")
df.head(10)

## 5. Exportacao dos dados

Exporta os resultados para um arquivo CSV para analise posterior.

In [ ]:
# Exporta para CSV
df.to_csv('levantamento.csv', index=False)

# Resumo dos resultados por orgao
print("Top 10 orgaos mais afetados:")
print(df['orgao'].value_counts().head(10))